# Cell 1:Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import shapiro
from sklearn.preprocessing import RobustScaler

# Load the dataset from the raw folder (relative to the notebooks folder)
df = pd.read_csv('../data/raw/full_leish_data_2010-2022.csv')

# Verify the shape to confirm all 156 rows are loaded
print(f"Initial Dataset Shape: {df.shape}")
df.head()

# Cell 2:Distribution Analysis for PRECT_TOTAL

In [ ]:
# Visualize the distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['PRECT_TOTAL'], kde=True, bins=30, color='blue')
plt.title('Distribution of PRECT_TOTAL')
plt.xlabel('Total Precipitation')
plt.ylabel('Frequency')
plt.show()

# Shapiro-Wilk test for normality
stat, p_value = shapiro(df['PRECT_TOTAL'].dropna())
print(f"Shapiro-Wilk Test - Statistic: {stat:.4f}, p-value: {p_value:.4f}")

if p_value > 0.05:
    print("Conclusion: PRECT_TOTAL looks normally distributed.")
else:
    print("Conclusion: PRECT_TOTAL is NOT normally distributed (likely skewed).")

# Cell 3:Outlier Handling (Winsorization)

In [ ]:
def clip_outliers(series, lower_percentile=0.05, upper_percentile=0.95):
    lower_bound = series.quantile(lower_percentile)
    upper_bound = series.quantile(upper_percentile)
    return series.clip(lower=lower_bound, upper=upper_bound)

# Define your continuous meteorological features
continuous_cols = ['RH2M', 'TMP_MIN', 'TMP_MAX', 'WS2M', 'WS10M', 'WD10M', 
                   'WD2M', 'UVA', 'UVB', 'CLOUD_AMT', 'SOIL_M', 'SOIL_W', 
                   'SOIL_RW', 'PRECT_TOTAL', 'SURFACE_P', 'SH2M']

# Apply clipping to preserve all rows
for col in continuous_cols:
    if col in df.columns:
        df[col] = clip_outliers(df[col])

print("Outliers clipped successfully. Row count remains:", len(df))

# Cell 4:Correlation Study & Parameter Reduction

In [ ]:
# Generate Spearman correlation matrix
corr_matrix = df[continuous_cols].corr(method='spearman')

# Plot the heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Spearman Correlation Heatmap')
plt.show()

# Automatically identify highly correlated pairs
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.85:
            colname_i = corr_matrix.columns[i]
            colname_j = corr_matrix.columns[j]
            high_corr_pairs.append((colname_i, colname_j, corr_matrix.iloc[i, j]))

print("Highly correlated pairs (Consider dropping one feature from each pair):")
for pair in high_corr_pairs:
    print(f"- {pair[0]} & {pair[1]}: {pair[2]:.2f}")

# Example lists of columns to drop based on typical weather data redundancy.
# IMPORTANT: Adjust 'cols_to_drop' based on the actual printed pairs above.
cols_to_drop = ['WS10M', 'WD10M', 'UVB', 'SOIL_W', 'SOIL_RW'] 
df_reduced = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

print(f"\nDataset shape after dropping redundant columns: {df_reduced.shape}")

# Cell 5: Scaling and PCA Variance Visualization

In [ ]:
# 1. Scaling (Mandatory before PCA)
scaler = RobustScaler()
scaled_features = scaler.fit_transform(df[continuous_cols])

# 2. Fit PCA to see variance distribution
pca_test = PCA()
pca_test.fit(scaled_features)

# 3. Plot Explained Variance
plt.figure(figsize=(10, 5))
plt.plot(np.cumsum(pca_test.explained_variance_ratio_), marker='o', linestyle='--')
plt.axhline(y=0.95, color='r', linestyle='-', label='95% Explained Variance')
plt.title('Cumulative Explained Variance by PCA Components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.legend(loc='best')
plt.grid(True)
plt.show()

# Cell 6: Final PCA Transformation and Dataset Assembly

In [ ]:
# Apply final PCA keeping exactly enough components to explain 95% of the variance
# Scikit-learn allows passing a float to n_components to automate this
pca_final = PCA(n_components=0.95)
pca_features = pca_final.fit_transform(scaled_features)

# Create a DataFrame for the new PCA components
df_pca = pd.DataFrame(
    pca_features, 
    columns=[f'PC{i+1}' for i in range(pca_features.shape[1])]
)

print(f"Number of components retained for 95% variance: {pca_features.shape[1]}")

# Reintegrate the non-continuous columns (Mois, Date, Cases)
columns_to_keep = [col for col in df.columns if col not in continuous_cols]
df_final = pd.concat([df[columns_to_keep].reset_index(drop=True), df_pca], axis=1)

print(f"Final dataset shape ready for Random Forest: {df_final.shape}")
df_final.head()

# Cellule 7 : Entraînement XGBoost et Diagnostics

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
import numpy as np

# 1. Définition de la cible (X et y)
# Remplacer 'Cases' par le nom exact de votre colonne cible si différent
X = df_final.drop(columns=['Cases', 'Mois', 'Date'], errors='ignore')
y = df_final['Cases']

# 2. Répartition 90:10 (Développement et Holdout)
X_dev, X_holdout, y_dev, y_holdout = train_test_split(X, y, test_size=0.10, stratify=y, random_state=42)
print(f"Taille de l'ensemble de développement (90%) : {X_dev.shape[0]} lignes")
print(f"Taille de l'ensemble de test/holdout (10%) : {X_holdout.shape[0]} lignes")

# 3. Calcul de la Baseline (pour le Biais Évitable)
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_dev, y_dev)
baseline_acc = dummy_clf.score(X_dev, y_dev)
baseline_error = 1 - baseline_acc

# 4. Configuration de XGBoost et de la grille d'hyperparamètres
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 150],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# 5. Recherche des meilleurs hyperparamètres avec Validation Croisée (10-Fold)
grid_search = GridSearchCV(
    estimator=xgb_model, 
    param_grid=param_grid, 
    cv=10, 
    scoring='accuracy', 
    n_jobs=-1,
    return_train_score=True # Crucial pour calculer le biais
)

print("\nRecherche des meilleurs hyperparamètres en cours...")
grid_search.fit(X_dev, y_dev)

best_model = grid_search.best_estimator_
print(f"\nMeilleurs hyperparamètres trouvés : {grid_search.best_params_}")

# 6. Extraction des métriques pour les diagnostics
best_index = grid_search.best_index_
cv_accuracy = grid_search.cv_results_['mean_test_score'][best_index]
train_accuracy = grid_search.cv_results_['mean_train_score'][best_index]

# Évaluation sur le sous-ensemble de test (10%)
holdout_predictions = best_model.predict(X_holdout)
holdout_accuracy = accuracy_score(y_holdout, holdout_predictions)

# 7. Calculs des diagnostics : Biais, Biais Évitable et Variance
train_error = 1 - train_accuracy
cv_error = 1 - cv_accuracy

bias = train_error
avoidable_bias = max(0, train_error - baseline_error)
variance = max(0, cv_error - train_error)

# 8. Affichage des diagnostics
print("\n--- DIAGNOSTICS DE PERFORMANCE ---")
print(f"Erreur de Base (Baseline Error) : {baseline_error:.4f} (Prédiction de la classe majoritaire)")
print(f"Erreur d'Entraînement : {train_error:.4f}")
print(f"Erreur de Validation Croisée : {cv_error:.4f}")
print(f"Erreur sur le Holdout (Test) : {1 - holdout_accuracy:.4f}")

print("\n--- ANALYSE BIAIS / VARIANCE ---")
print(f"Biais Total : {bias:.4f}")
print(f"Biais Évitable (Avoidable Bias) : {avoidable_bias:.4f}")
print(f"Variance : {variance:.4f}")

if avoidable_bias > variance:
    print("\nDiagnostic : Le modèle souffre d'un SOUS-APPRENTISSAGE (Underfitting).")
    print("Action recommandée : Augmenter la complexité du modèle (max_depth plus élevé), réduire la régularisation, ou ajouter des caractéristiques polynomiales.")
elif variance > avoidable_bias:
    print("\nDiagnostic : Le modèle souffre d'un SUR-APPRENTISSAGE (Overfitting).")
    print("Action recommandée : Augmenter la régularisation (gamma, alpha, lambda), réduire max_depth, ou collecter plus de données réelles.")
else:
    print("\nDiagnostic : Le modèle présente un bon équilibre entre biais et variance.")